# Strategy 3: PPO/DDQN training on Colab CPU

Runs `Strategy 3`'s pipeline: PPO/DDQN training (ASU as a low-probability
training opponent only) -> seat-balanced evaluation against Fixed-A/B/C.

Read `Strategy 3/PLAN.md` first, especially §2 and §9's rule-compliance note.
The MonopolyZero/`monopoly_bench` self-play track was dropped: its `Trainer`
bootstraps the policy head by cross-entropy against ASU's chosen actions
unconditionally on every fresh run, with no way to disable it through the
public API. That is training on ASU's output, which the competition rules
(2026-08-10/11, "ASU'yu birebir output klonlamak yasak") and this repo's own
`CLAUDE.md` both forbid outside the SLM/Gemma track. Two settled points this
notebook assumes:

- **Metric (PLAN.md §1)**: baseline-relative. The number that matters is this
  checkpoint's seat-balanced win rate against Fixed-A/B/C, compared against
  the measured `asu_value_v1` baseline of **72/100** (`Strategy 1/REPO_STUDY_NOTES.md`).
  ASU is never seated as an opponent in the final evaluation cell.
- **ASU's role (PLAN.md §2, corrected)**: opponent seat only, at low
  probability (`--asu-opponent-probability`). Never a training target.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/DeepRL_Monopoly'
!mkdir -p "$DRIVE_ROOT/artifacts"

## Get the code

Clones from the `Gokturkakman/DeepRL_Monopoly` fork (no write access to
`Darkosxl/DeepRL_Monopoly` upstream), branch `feature/strategy-3-hybrid` --
`Strategy 3` hasn't been merged to `main` yet. If you get a 404 or a tree
without `Strategy 3/`, either the branch was renamed/merged since this was
written (update `BRANCH` below) or it hasn't been pushed yet.

In [ ]:
REPO_URL = 'https://github.com/Gokturkakman/DeepRL_Monopoly.git'  # fork -- no write access to Darkosxl/DeepRL_Monopoly
BRANCH = 'feature/strategy-3-hybrid'  # not merged to main yet

!rm -rf /content/DeepRL_Monopoly
!git clone --branch $BRANCH --depth 1 $REPO_URL /content/DeepRL_Monopoly
%cd "/content/DeepRL_Monopoly/Strategy 3"
!ls

In [ ]:
# Colab ships torch + numpy preinstalled and both already satisfy this
# repo's requirements.txt (numpy>=1.26) -- no upgrade needed. (An earlier
# version of this cell ran `pip install --upgrade numpy`, which only
# confused the version print below -- the upgraded wheel lands on disk but
# this kernel already had numpy cached from Colab's own startup, so the
# print kept showing the old version until a runtime restart. Training
# cells below spawn fresh `!python` subprocesses regardless, which would
# have picked up the upgrade either way -- but skipping it avoids the
# numba<2.5 dependency-conflict warning Colab's preinstalled numba throws.)
import torch, numpy
print('torch', torch.__version__, 'cuda available (unused here):', torch.cuda.is_available())
print('numpy', numpy.__version__)

## Step 1 -- PPO + DDQN in parallel

The TPU v5e-1 runtime type gives a much bigger vCPU allocation than the plain
CPU runtime -- the game-simulation loop itself won't use the TPU chip
(nothing in this repo does; see PLAN.md's compute notes), but with more
cores available it's worth running the two independent training runs as
separate background processes instead of one after another.

`--asu-opponent-probability 0.15` (both runs) puts ASU in an opponent seat
15% of games -- raised from 0.02 so the model actually trains against ASU
often enough to matter, not just occasionally. Still legal: ASU's chosen
actions are never recorded as a training target, only the game outcome
feeds back (`train.py`'s `_sample_opponents`). Cost of the raise: ASU's own
decision cost is 1-2 orders of magnitude above a fixed heuristic's
(`--asu-decision-timeout 5.0` bounds the tail per decision), so more ASU
games means more wall-clock for the same game count.

`--self-play-probability` (DDQN only) is a *separate* mechanism layered on
top of the ASU opponent seat, not instead of it -- past DDQN snapshots as
additional opponents, pure self-play against its own history, no ASU
involvement in that specific piece. DDQN also gets the diagnosed short-run
fix -- default `lr=1e-5` / `target_update_freq=500 games` are tuned for a
10,000-game paper run; a 1000-game run at those defaults stayed at 0% win
rate with correctly-signed rewards throughout (`CLAUDE.md`, "Known-hard
problem").</cell id="cell-5">


In [ ]:
import subprocess

PPO_OUT = f'{DRIVE_ROOT}/artifacts/ppo_plus/ppo_hybrid_2000_v2.pt'
DDQN_OUT = f'{DRIVE_ROOT}/artifacts/ddqn_plus/ddqn_hybrid_colab.pt'
ppo_log = '/content/ppo_train.log'
ddqn_log = '/content/ddqn_train.log'

ppo_cmd = [
    'python', 'tools/train_and_save.py',
    '--algo', 'ppo', '--hybrid',
    '--games', '2000',
    '--device', 'cpu',
    '--seed', '42',
    '--checkpoint-every', '100',
    '--opponent-epsilon', '0.1',
    '--opponent-threshold-jitter', '0.2',
    '--held-out-eval-games', '20',
    '--asu-opponent-probability', '0.15',
    '--out', PPO_OUT,
]
ddqn_cmd = [
    'python', 'tools/train_and_save.py',
    '--algo', 'ddqn', '--hybrid',
    '--games', '2000',
    '--device', 'cpu',
    '--seed', '42',
    '--checkpoint-every', '100',
    '--lr', '1e-4',
    '--target-update-freq-steps', '2000',
    '--epsilon-decay', '0.9985',
    '--opponent-epsilon', '0.1',
    '--opponent-threshold-jitter', '0.2',
    '--held-out-eval-games', '20',
    '--asu-opponent-probability', '0.15',
    '--self-play-probability', '0.15',
    '--self-play-pool-size', '8',
    '--self-play-register-every', '200',
    '--out', DDQN_OUT,
]

ppo_proc = subprocess.Popen(ppo_cmd, stdout=open(ppo_log, 'w'), stderr=subprocess.STDOUT)
ddqn_proc = subprocess.Popen(ddqn_cmd, stdout=open(ddqn_log, 'w'), stderr=subprocess.STDOUT)
print('PPO started, pid', ppo_proc.pid, '-> log:', ppo_log)
print('DDQN started, pid', ddqn_proc.pid, '-> log:', ddqn_log)
print('Run the next cell to wait for both and tail their logs.')

Wait for both background runs, printing a status line every minute so the
tab stays visibly active. Safe to interrupt and re-run -- it only polls the
already-running processes, it doesn't restart them.

In [ ]:
import time

while ppo_proc.poll() is None or ddqn_proc.poll() is None:
    print(f"[{time.strftime('%H:%M:%S')}] PPO running={ppo_proc.poll() is None}  DDQN running={ddqn_proc.poll() is None}")
    time.sleep(60)

print('PPO exit code:', ppo_proc.returncode)
print('DDQN exit code:', ddqn_proc.returncode)
print('\n--- PPO log (last 30 lines) ---')
!tail -n 30 {ppo_log}
print('\n--- DDQN log (last 30 lines) ---')
!tail -n 30 {ddqn_log}

## Step 2 -- the metric that matters (PLAN.md §1)

Seat-balanced win rate against Fixed-A/B/C, with a Wilson interval
(`tools/evaluate_vs_fixed.py`, built on `ASU_FROZEN_TEACHER.evaluate.evaluate_lineup`
-- the exact function that produced the measured `asu_value_v1` baseline of
**72/100**, so this is the same code path, not just the same statistic).
ASU is not an opponent in this cell -- per PLAN.md §1 this is a
baseline-relative comparison, not a head-to-head against ASU.

In [ ]:
!python tools/evaluate_vs_fixed.py \
  --checkpoint "ppo:$PPO_OUT" \
  --seeds 25 \
  --out "$DRIVE_ROOT/artifacts/ppo_plus/eval_vs_fixed_abc_100.json"

!python tools/evaluate_vs_fixed.py \
  --checkpoint "ddqn:$DDQN_OUT" \
  --seeds 25 \
  --out "$DRIVE_ROOT/artifacts/ddqn_plus/eval_vs_fixed_abc_100.json"

Checkpoints and eval JSON all live under `$DRIVE_ROOT` on Drive, so they
survive the Colab VM being recycled. Pull anything back to your laptop by
downloading it from Drive directly -- don't route it through git;
`artifacts/` is gitignored on purpose.